In [1]:
!pip install chromadb==0.4.18 pypdf

In [2]:
!pip install langchain-text-splitters

First, let's create a dummy driving test document to ingest. In a real scenario, you would replace this with your actual document loading logic (e.g., `PyPDFLoader` for PDF files).

In [3]:
document_content = """
# Dummy Driving Test Document

## Section 1: Road Signs
- Stop signs are red and octagonal.
- Yield signs are red and inverted triangles.
- Speed limit signs are white and rectangular.

## Section 2: Traffic Laws
- Always stop at red lights.
- Yield to pedestrians in crosswalks.
- Do not text and drive.

## Section 3: Parking Rules
- Do not park within 15 feet of a fire hydrant.
- Parallel parking requires signaling and checking mirrors.
"""
with open("driving_test_document.md", "w") as f:
    f.write(document_content)

document_path = "driving_test_document.md"
print(f"Dummy document saved to {document_path}")

Dummy document saved to driving_test_document.md


Now, let's set up the LangChain components: load the document, split it into chunks, generate embeddings, and store them in Chroma.

In [4]:
!pip install langchain-community langchain langchain_classic
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_classic.chains import RetrievalQA
import os
from google.colab import userdata

# Ensure the API key is configured for all operations
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# Load the document
loader = TextLoader(document_path)
documents = loader.load()

# Split the document into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(documents)
print(f"Split {len(documents)} document into {len(splits)} chunks.")

/tmp/ipykernel_7404/3815956454.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


Split 1 document into 1 chunks.


Next, we'll initialize the embedding model and create the Chroma vector store.

In [5]:
# Initialize Google Generative AI Embeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# To resolve numpy compatibility issue, downgrade numpy if necessary.
# This error usually means a dependency expects numpy < 2.0
!pip install numpy==1.26.4

# Create Chroma vector store
# We'll persist the store to disk for later use
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings, persist_directory="./chroma_db")
print("Chroma vector store created and persisted to ./chroma_db")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Chroma vector store created and persisted to ./chroma_db


In [6]:
import google.generativeai as genai
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

print("Available models that support embedContent:")
for m in genai.list_models():
  if "embedContent" in m.supported_generation_methods:
    print(m.name)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Available models that support embedContent:
models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


Now, let's set up the `gemini-2.5-pro` LLM and the LangChain Retrieval QA chain to answer questions based on the ingested document.

In [7]:
# Initialize the Gemini LLM with gemini-2.5-flash
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

# Create a retrieval chain
qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=vectorstore.as_retriever())
print("Retrieval QA chain created.")

Retrieval QA chain created.


Finally, let's ask a question and get a response from the LLM based on the ingested document.

In [8]:
# Ask a question based on the document
query = "What are the rules about parking near a fire hydrant?"
response = qa_chain.invoke({"query": query})
print("\nQuery:", query)
print("Response:", response["result"])

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



Query: What are the rules about parking near a fire hydrant?
Response: Do not park within 15 feet of a fire hydrant.
